# Alexandria Audiobook Generator — Google Colab

Run [Alexandria](https://github.com/Finrandojin/alexandria-audiobook) on a free Google Colab GPU. No local installation required.

> **Note:** This Colab notebook is provided for convenience so you can try Alexandria without local installation. It is not the primary focus of the project — for the best experience, install Alexandria locally via [Pinokio](https://pinokio.computer).

**What this notebook does:**
1. Checks your GPU runtime
2. Mounts Google Drive so Alexandria's files, output, and TTS models persist across sessions
3. Installs Alexandria and all dependencies
4. Starts the Alexandria server and opens the web UI inline (no ngrok needed)

**Before you start:**
- Make sure you've selected a **GPU runtime**: Runtime → Change runtime type → T4 GPU
- You need a Google account (for Drive) and an LLM server for script generation (see Cell 5 for options)

**Colab T4 GPU (15 GB VRAM) recommended settings:**
- Parallel Workers: 5-10
- Max Chars/Batch: 1500-2000
- Compile Codec: enabled (recommended for speed)

---

## 1. Check GPU Runtime

Verify that a GPU is available. If this cell fails, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → T4 GPU, then re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print()
if vram_gb < 10:
    print("WARNING: Less than 10 GB VRAM. Batch sizes will be very limited.")
    print("Use Parallel Workers: 2-3 and Max Chars/Batch: 1000 in Setup.")
elif vram_gb < 16:
    print("T4 detected (15 GB). Recommended: Parallel Workers 5-10, Max Chars/Batch 1500-2000.")
else:
    print(f"{vram_gb:.0f} GB VRAM available. You can use higher batch sizes.")

## 2. Mount Google Drive

Alexandria stores its code, generated audio, uploaded books, voice configs, and TTS models on Google Drive so **everything persists across Colab sessions**. TTS models alone are ~3.5 GB each; without Drive, they re-download every time.

This cell:
- Mounts your Drive at `/content/drive`
- Creates `MyDrive/Alexandria` (if it doesn't exist)
- Symlinks `/content/Alexandria` → the Drive folder, so the rest of the notebook just uses the local path
- Points HuggingFace's cache at Drive too, so downloaded models persist

You'll be prompted to authorize Colab to access your Drive.

In [ ]:
import os

from google.colab import drive

drive_path = "/content/drive/MyDrive/Alexandria"
local_path = "/content/Alexandria"

drive.mount("/content/drive")

# Create the Drive directory on first use
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"Created Drive directory: {drive_path}")

# Symlink /content/Alexandria -> Drive so subsequent cells just use the local path
if not os.path.exists(local_path):
    os.symlink(drive_path, local_path)
    print(f"Symlink: {local_path} -> {drive_path}")
elif os.path.islink(local_path):
    print(f"Symlink already in place: {local_path} -> {os.readlink(local_path)}")
else:
    print(
        f"WARNING: {local_path} already exists and is not a symlink. "
        "Delete it manually if you want Drive-backed storage."
    )

# Persist HuggingFace model cache on Drive too
hf_cache = "/content/drive/MyDrive/.cache/huggingface"
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"] = hf_cache

print(f"HuggingFace cache: {hf_cache}")
print("Alexandria files and TTS models will persist across sessions.")

## 3. Install Alexandria

Clones the repository into your Drive-backed Alexandria folder and installs all Python dependencies. First run takes 2-3 minutes; subsequent runs are much faster since the clone already exists on Drive.

In [ ]:
import os

ALEXANDRIA_DIR = "/content/Alexandria"

# Clone or update the repository (the target is symlinked to Drive)
if not os.path.exists(os.path.join(ALEXANDRIA_DIR, ".git")):
    !git clone https://github.com/Finrandojin/alexandria-audiobook.git {ALEXANDRIA_DIR}
else:
    print(f"Alexandria already cloned at {ALEXANDRIA_DIR}")
    !cd {ALEXANDRIA_DIR} && git pull

# Install dependencies (skip torch — Colab already has it)
!pip install -q -r {ALEXANDRIA_DIR}/app/requirements.txt
!pip install -q qwen-tts==0.1.1

print()
print("Installation complete.")

## 4. Start Alexandria

This cell:
1. Writes a default config if none exists (local TTS mode, auto GPU device)
2. Starts the Alexandria server in the background
3. Displays the web UI inline via Colab's built-in port forwarding — **no ngrok, no auth token, no signup**

After the server starts, an inline iframe appears below with the Alexandria UI. If you'd rather open it in its own tab, use the "Open in new tab" link that also prints below.

### LLM Setup

Alexandria needs an LLM server for script generation. In the web UI **Setup tab**, configure one of:

| Provider | Base URL | API Key |
|----------|----------|--------|
| OpenAI | `https://api.openai.com/v1` | Your OpenAI API key |
| DeepSeek | `https://api.deepseek.com/v1` | Your DeepSeek API key |
| OpenRouter | `https://openrouter.ai/api/v1` | Your OpenRouter API key |
| Ollama (see Cell 6) | `http://localhost:11434/v1` | `local` |

Or any other OpenAI-compatible API.

In [ ]:
import json
import os
import subprocess
import time

import requests
from google.colab import output

ALEXANDRIA_DIR = "/content/Alexandria"
APP_DIR = os.path.join(ALEXANDRIA_DIR, "app")
CONFIG_PATH = os.path.join(APP_DIR, "config.json")

# Write default config if none exists (persists on Drive)
if not os.path.exists(CONFIG_PATH):
    config = {
        "llm": {
            "base_url": "http://localhost:11434/v1",
            "api_key": "local",
            "model_name": ""
        },
        "tts": {
            "mode": "local",
            "device": "auto",
            "language": "English",
            "parallel_workers": 8,
            "compile_codec": True,
            "sub_batch_enabled": True,
            "sub_batch_min_size": 4,
            "sub_batch_ratio": 5,
            "sub_batch_max_chars": 2000
        }
    }
    with open(CONFIG_PATH, "w") as f:
        json.dump(config, f, indent=2)
    print("Default config written (local TTS, auto GPU).")
else:
    print("Existing config.json found on Drive, keeping it.")

# Start the server
print("Starting Alexandria server...")
server_process = subprocess.Popen(
    ["python", "app.py"],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait for server to start
for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:4200/api/config", timeout=2)
        if r.status_code == 200:
            print("Server is running.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print("WARNING: Server may not have started. Check output below.")

# Show a "open in new tab" link and the embedded iframe below it
print()
print("=" * 60)
print("Alexandria is ready. Use the inline iframe below, or:")
output.serve_kernel_port_as_window(4200, anchor_text="↗ Open Alexandria in a new tab")
print("=" * 60)
print()
print("First TTS generation will download the model (~3.5 GB).")
print("It's cached to Drive so future sessions skip the download.")
print("Check this cell's output for progress.")

# Inline embed
output.serve_kernel_port_as_iframe(4200, height=800)

## 5. (Optional) Install Ollama for Local LLM

If you don't have a cloud LLM API, you can run Ollama on Colab for script generation.

**Important — VRAM sharing:** Ollama and TTS both use the T4 GPU. A 7B model uses ~5 GB VRAM, leaving only ~10 GB for TTS. This **will cause out-of-memory crashes** during batch generation, especially with LoRA voices.

**Workflow:** Use Ollama for script generation, then **run Cell 5b to stop Ollama** before starting TTS batch generation. The LLM is only needed during script generation.

**Skip this cell** if you're using a cloud API (OpenAI, DeepSeek, etc.) — that's the easiest option on Colab.

In [ ]:
import subprocess
import time

OLLAMA_MODEL = "qwen2.5:7b"  # @param {type:"string"}

# Install zstd (required by Ollama installer)
!apt-get install -y -qq zstd

# Install Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
print("Starting Ollama server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# Pull the model
print(f"Pulling {OLLAMA_MODEL} (this may take a few minutes)...")
!ollama pull {OLLAMA_MODEL}

print()
print(f"Ollama is running with {OLLAMA_MODEL}.")
print()
print("In Alexandria Setup tab, configure:")
print("  LLM Base URL: http://localhost:11434/v1")
print("  API Key: local")
print(f"  Model Name: {OLLAMA_MODEL}")

## 5b. Stop Ollama (Free VRAM for TTS)

**Run this cell after script generation is complete**, before starting batch TTS generation. This frees ~5 GB of VRAM that Ollama was using for the LLM.

You do NOT need to run this if you used a cloud LLM API instead of Ollama.

In [ ]:
import subprocess

# Stop Ollama server to free GPU memory for TTS
try:
    subprocess.run(["pkill", "-f", "ollama"], timeout=5, check=False)
    print("Ollama stopped. GPU memory freed for TTS.")
except (subprocess.SubprocessError, OSError):
    print("Ollama was not running.")

# Verify VRAM is freed
import torch

if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    total_mem = torch.cuda.mem_get_info()[1] / 1e9
    print(f"VRAM: {free_mem:.1f} GB free / {total_mem:.1f} GB total")

## 6. View Server Logs

Run this cell to see real-time server output (model loading, generation progress, errors).

**Interrupt the cell** (stop button) to stop viewing logs. The server keeps running.

In [ ]:
# Stream server output (interrupt to stop viewing, server keeps running)
try:
    for line in server_process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nStopped viewing logs. Server is still running.")

## 7. Stop Server

Run this cell when you're done to clean up.

In [ ]:
# Stop the Alexandria server
try:
    server_process.terminate()
    server_process.wait(timeout=5)
    print("Server stopped.")
except (ProcessLookupError, subprocess.TimeoutExpired):
    server_process.kill()
    print("Server killed.")

# Stop Ollama if it was running
try:
    ollama_process.terminate()
    print("Ollama stopped.")
except (ProcessLookupError, subprocess.TimeoutExpired):
    pass